# STM32F429 Ethernet (ETH) Peripheral Tutorial

This comprehensive tutorial covers the Ethernet peripheral on the STM32F429 microcontroller. The Ethernet MAC (Media Access Controller) provides 10/100 Mbps Ethernet connectivity with hardware acceleration for network communications.

## Table of Contents
1. [Ethernet Overview](#ethernet-overview)
2. [Hardware Architecture](#hardware-architecture)
3. [Register Configuration](#register-configuration)
4. [API Reference](#api-reference)
5. [Basic Usage Examples](#basic-usage-examples)
6. [Advanced Features](#advanced-features)
7. [Network Protocols](#network-protocols)
8. [Performance Considerations](#performance-considerations)
9. [Troubleshooting](#troubleshooting)

---

## Ethernet Overview

The STM32F429 Ethernet peripheral provides complete hardware support for 10/100 Mbps Ethernet networking:

### Key Features
- **IEEE 802.3 Compliant MAC**: Full Media Access Controller implementation
- **10/100 Mbps Support**: Auto-negotiation and manual speed/duplex configuration
- **Hardware Checksum Offload**: TCP/UDP/IP checksum calculation in hardware
- **DMA Support**: Direct memory access for high-performance data transfers
- **Flexible Buffer Management**: Configurable transmit/receive buffers
- **Interrupt Support**: Multiple interrupt sources for efficient processing
- **VLAN Support**: IEEE 802.1Q VLAN tagging
- **RMII/MII Interface**: Reduced/Standard Media Independent Interface
- **PHY Support**: Integrated support for various Ethernet PHYs

### Applications
- Embedded web servers
- Network-enabled IoT devices
- Industrial Ethernet communications
- Network monitoring and control systems
- Firmware update over network (TFTP)
- Remote monitoring and diagnostics
- Real-time data streaming
- Network protocol implementation

### Performance Benefits
- **Hardware Acceleration**: Offloads CPU from network processing
- **Low Latency**: Direct memory access minimizes interrupt overhead
- **High Throughput**: Up to 100 Mbps full-duplex operation
- **Efficient Buffers**: Optimized memory usage for network packets
- **Interrupt Optimization**: Configurable interrupt thresholds
- **Checksum Offload**: Reduces CPU load for protocol processing

### Supported Protocols
- **Ethernet II**: Standard Ethernet framing
- **IEEE 802.1Q VLAN**: Virtual LAN tagging
- **ARP**: Address Resolution Protocol
- **IPv4/IPv6**: Internet Protocol support
- **TCP/UDP**: Transport layer protocols
- **ICMP**: Internet Control Message Protocol
- **DHCP**: Dynamic Host Configuration Protocol
- **DNS**: Domain Name System

## Hardware Architecture

### STM32F429 Ethernet Block Diagram

```
PHY Chip (e.g., LAN8720, DP83848)
   |
   | RMII/MII Interface
   v
   +---------------------+
   |     Ethernet MAC    |
   |                     |
   |  +---------------+  |
   |  |   MAC Core    |  |
   |  +---------------+  |
   |         |           |
   |  +---------------+  |
   |  |   DMA Engine  |  |
   |  +---------------+  |
   +---------------------+
             |
             | AHB Bus
             v
        System Memory
        (TX/RX Buffers)
```

### Key Components
- **MAC (Media Access Controller)**: Implements Ethernet protocol logic
- **DMA Engine**: Handles data transfers between MAC and system memory
- **PHY Interface**: RMII (Reduced MII) or MII connection to external PHY
- **Buffer Descriptors**: Linked lists managing transmit/receive buffers
- **Statistics Counters**: Network traffic statistics and error counters
- **Control Registers**: Configuration and status registers

### Memory Mapping
- **Base Address**: 0x40028000
- **Size**: 0x1400 bytes (5KB)
- **Bus**: AHB1
- **Interrupt**: ETH_IRQn (IRQ #61)
- **Wakeup Interrupt**: ETH_WKUP_IRQn (IRQ #62)

### Buffer Management
- **TX Descriptors**: Control transmit buffer management
- **RX Descriptors**: Control receive buffer management
- **TX Buffers**: Memory areas for outgoing packets
- **RX Buffers**: Memory areas for incoming packets
- **Descriptor Tables**: Arrays of descriptor structures

### PHY Interface Options
- **RMII (Reduced MII)**: 7 signals, lower pin count
- **MII (Media Independent Interface)**: 18 signals, full interface
- **Auto-negotiation**: Automatic speed and duplex detection
- **Manual Configuration**: Fixed speed/duplex settings

## Register Configuration

### Ethernet MAC Registers Overview

The STM32F429 Ethernet peripheral uses multiple register groups for configuration and control:

#### MAC Configuration Register (MACCR) - 0x40028000
- **Purpose**: Main MAC configuration register
- **Access**: Read/Write
- **Reset Value**: 0x00008000
- **Bit Fields**:
  - Bit 0: RE - Receiver enable
  - Bit 1: TE - Transmitter enable
  - Bit 2: PRELEN[1:0] - Preamble length
  - Bit 3: DC - Deferral check
  - Bit 4: BL[1:0] - Back-off limit
  - Bit 5: APCS - Automatic pad/CRC stripping
  - Bit 6: RD - Retry disable
  - Bit 7: IPCO - IPv4 checksum offload
  - Bit 8: DM - Duplex mode
  - Bit 9: LM - Loopback mode
  - Bit 10: ROD - Receive own disable
  - Bit 11: FES - Fast Ethernet speed
  - Bit 12: CSD - Carrier sense disable
  - Bit 13: IFG[2:0] - Inter-frame gap
  - Bit 16: JD - Jabber disable
  - Bit 17: WD - Watchdog disable
  - Bits[31:18]: Reserved

#### MAC Frame Filter Register (MACFFR) - 0x40028004
- **Purpose**: Controls frame filtering and reception
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bit 0: PM - Promiscuous mode
  - Bit 1: HU - Hash unicast
  - Bit 2: HM - Hash multicast
  - Bit 3: DAIF - Destination address inverse filtering
  - Bit 4: PAM - Pass all multicast
  - Bit 5: BFD - Broadcast frames disable
  - Bit 6: PCF[1:0] - Pass control frames
  - Bit 8: SAIF - Source address inverse filtering
  - Bit 9: SAF - Source address filter
  - Bit 10: HPF - Hash or perfect filter
  - Bit 31: RA - Receive all

#### MAC Hash Table High Register (MACHTHR) - 0x40028008
- **Purpose**: Upper 32 bits of hash table for multicast filtering
- **Access**: Read/Write
- **Reset Value**: 0x00000000

#### MAC Hash Table Low Register (MACHTLR) - 0x4002800C
- **Purpose**: Lower 32 bits of hash table for multicast filtering
- **Access**: Read/Write
- **Reset Value**: 0x00000000

#### MAC MII Address Register (MACMIIAR) - 0x40028010
- **Purpose**: Controls MII management interface
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[4:0]: MB - MII busy
  - Bits[7:5]: MW - MII write
  - Bits[12:8]: MR[4:0] - MII register
  - Bits[17:16]: CR[1:0] - Clock range
  - Bits[31:18]: PA[4:0] - PHY address

#### MAC MII Data Register (MACMIIDR) - 0x40028014
- **Purpose**: MII data register for PHY communication
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[15:0]: MD - MII data

#### MAC Flow Control Register (MACFCR) - 0x40028018
- **Purpose**: Controls flow control operation
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bit 0: FCB - Flow control busy/back pressure activate
  - Bit 1: TFE - Transmit flow control enable
  - Bit 2: RFE - Receive flow control enable
  - Bit 3: UP - Unicast pause frame detect
  - Bits[7:4]: PLT[2:0] - Pause low threshold
  - Bits[15:8]: ZQPD - Zero-quanta pause disable
  - Bits[31:16]: PT[15:0] - Pause time

#### MAC VLAN Tag Register (MACVLANTR) - 0x4002801C
- **Purpose**: VLAN tag comparison and insertion
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[15:0]: VLANTI - VLAN tag identifier
  - Bits[17:16]: VLANTC - VLAN tag comparison

#### MAC PMT Control and Status Register (MACPMTCSR) - 0x4002802C
- **Purpose**: Power management control
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bit 0: PD - Power down
  - Bit 1: MPE - Magic packet enable
  - Bit 2: WFE - Wakeup frame enable
  - Bits[4:3]: MPR - Magic packet received
  - Bits[6:5]: WFR - Wakeup frame received
  - Bit 7: GU - Global unicast
  - Bits[31:8]: WFFRPR - Wakeup frame filter register pointer

#### MAC Interrupt Status Register (MACISR) - 0x40028038
- **Purpose**: MAC interrupt status
- **Access**: Read only
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bit 0: PMTS - PMT status
  - Bit 1: MMCS - MMC status
  - Bit 2: MMCRS - MMC receive status
  - Bit 3: MMCTS - MMC transmit status
  - Bit 5: TS - Timestamp status
  - Bits[31:6]: Reserved

#### MAC Interrupt Mask Register (MACIMR) - 0x4002803C
- **Purpose**: MAC interrupt mask
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bit 0: PMTIM - PMT interrupt mask
  - Bit 1: Reserved
  - Bit 2: Reserved
  - Bit 3: Reserved
  - Bit 5: TSIM - Timestamp interrupt mask

### DMA Registers

#### DMA Bus Mode Register (DMABMR) - 0x40029000
- **Purpose**: DMA bus mode configuration
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bit 0: SR - Software reset
  - Bit 1: DA - DMA arbitration
  - Bits[6:2]: DSL[4:0] - Descriptor skip length
  - Bit 7: EDFE - Enhanced descriptor format enable
  - Bits[13:8]: PBL[5:0] - Programmable burst length
  - Bit 14: RTPR[1:0] - RX/TX priority ratio
  - Bit 16: FB - Fixed burst
  - Bit 17: RDP[5:0] - RX DMA PBL
  - Bit 22: USP - Use separate PBL
  - Bit 23: FPM - Four TX queues
  - Bit 24: AAB - Address-aligned beats

#### DMA Status Register (DMASR) - 0x40029004
- **Purpose**: DMA status and interrupt flags
- **Access**: Read only (some bits write 1 to clear)
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[2:0]: TS - Transmit status
  - Bits[6:3]: RS - Receive status
  - Bit 7: NIS - Normal interrupt summary
  - Bit 8: AIS - Abnormal interrupt summary
  - Bit 9: ERI - Early receive interrupt
  - Bit 10: FBI - Fatal bus error interrupt
  - Bit 11: ETI - Early transmit interrupt
  - Bit 12: RWT - Receive watchdog timeout
  - Bit 13: RPS - Receive process stopped
  - Bit 14: RBU - Receive buffer unavailable
  - Bit 15: RI - Receive interrupt
  - Bit 16: TBU - Transmit buffer unavailable
  - Bit 17: TPS - Transmit process stopped
  - Bit 18: TI - Transmit interrupt

#### DMA Operation Mode Register (DMAOMR) - 0x40029008
- **Purpose**: DMA operation mode
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bit 1: SR - Start/stop receive
  - Bit 2: OSF - Operate on second frame
  - Bit 3: RTC[1:0] - Receive threshold control
  - Bit 6: FUGF - Forward undersized good frames
  - Bit 7: FEF - Forward error frames
  - Bit 8: ST - Start/stop transmission
  - Bit 9: TTC[2:0] - Transmit threshold control
  - Bit 13: FTF - Flush transmit FIFO
  - Bit 14: TSF - Transmit store and forward
  - Bit 20: DFRF - Disable flushing of received frames
  - Bit 21: RSF - Receive store and forward
  - Bit 24: DTCEFD - Dropping of TCP/IP checksum error frames disable

### Register Setup Sequence

```c
// Enable Ethernet clocks
RCC->AHB1ENR |= RCC_AHB1ENR_ETHMACEN | RCC_AHB1ENR_ETHMACTXEN | RCC_AHB1ENR_ETHMACRXEN;

// Reset Ethernet MAC
ETH->DMABMR |= ETH_DMABMR_SR;
while (ETH->DMABMR & ETH_DMABMR_SR);

// Configure MAC
ETH->MACCR = ETH_MACCR_FES | ETH_MACCR_DM | ETH_MACCR_APCS | ETH_MACCR_IPCO;
ETH->MACFFR = ETH_MACFFR_HPF;  // Use perfect filter

// Set MAC address (example: 00:11:22:33:44:55)
ETH->MACA0HR = 0x00112233;
ETH->MACA0LR = 0x44550000;

// Configure DMA
ETH->DMABMR = ETH_DMABMR_AAB | ETH_DMABMR_FB | (32 << 8);  // Burst length 32
ETH->DMAOMR = ETH_DMAOMR_RSF | ETH_DMAOMR_TSF | ETH_DMAOMR_ST | ETH_DMAOMR_SR;

// Enable interrupts
ETH->DMAIER = ETH_DMAIER_NISE | ETH_DMAIER_RIE | ETH_DMAIER_TIE;

// Start Ethernet
ETH->MACCR |= ETH_MACCR_TE | ETH_MACCR_RE;
```

### PHY Configuration

**MII Management Interface Setup:**
- **PHY Address**: 0-31 (typically 0 or 1)
- **Clock Divider**: Based on HCLK frequency
- **Register Access**: Read/write PHY registers

**Auto-negotiation Process:**
1. Reset PHY
2. Enable auto-negotiation
3. Wait for completion
4. Read link status
5. Configure MAC accordingly

**Manual Configuration:**
- Set fixed speed (10/100 Mbps)
- Set duplex mode (half/full)
- Disable auto-negotiation

## API Reference

### Configuration Structures

```c
typedef struct {
    uint8_t macAddr[6];              // MAC address (6 bytes)
    uint32_t speed;                  // ETH_SPEED_10M or ETH_SPEED_100M
    uint32_t duplexMode;             // ETH_MODE_FULLDUPLEX or ETH_MODE_HALFDUPLEX
    uint32_t checksumMode;           // ETH_CHECKSUM_BY_HARDWARE or ETH_CHECKSUM_BY_SOFTWARE
    uint32_t mediaInterface;         // ETH_MEDIA_INTERFACE_RMII or ETH_MEDIA_INTERFACE_MII
    uint32_t vlanTagIdentifier;      // VLAN tag identifier (0 = disabled)
    uint32_t vlanTagProtocol;        // VLAN tag protocol identifier
} ETH_Config_t;

typedef struct {
    ETH_HandleTypeDef heth;          // HAL ETH handle
    ETH_Config_t config;             // Configuration
    bool initialized;                // Initialization status
    uint8_t *rxBuffer;               // RX buffer pointer
    uint32_t rxBufferSize;           // RX buffer size
    uint8_t *txBuffer;               // TX buffer pointer
    uint32_t txBufferSize;           // TX buffer size
} ETH_Handle_t;

typedef struct {
    uint8_t destination[6];          // Destination MAC address
    uint8_t source[6];               // Source MAC address
    uint16_t type;                   // Ethernet type (0x0800=IPv4, 0x0806=ARP, etc.)
    uint8_t *payload;                // Payload data pointer
    uint32_t payloadLength;          // Payload length in bytes
} ETH_Frame_t;
```

### Core Functions

#### Initialization and Control
```c
HAL_StatusTypeDef ETH_Init(ETH_Handle_t *handle, ETH_Config_t *config);
HAL_StatusTypeDef ETH_DeInit(ETH_Handle_t *handle);
HAL_StatusTypeDef ETH_Start(ETH_Handle_t *handle);
HAL_StatusTypeDef ETH_Stop(ETH_Handle_t *handle);
```

#### Data Transmission
```c
HAL_StatusTypeDef ETH_TransmitFrame(ETH_Handle_t *handle, ETH_Frame_t *frame);
HAL_StatusTypeDef ETH_TransmitFrame_IT(ETH_Handle_t *handle, ETH_Frame_t *frame);
HAL_StatusTypeDef ETH_TransmitFrame_DMA(ETH_Handle_t *handle, uint8_t *frame, uint32_t length);
```

#### Data Reception
```c
HAL_StatusTypeDef ETH_ReceiveFrame(ETH_Handle_t *handle, ETH_Frame_t *frame);
HAL_StatusTypeDef ETH_ReceiveFrame_IT(ETH_Handle_t *handle, ETH_Frame_t *frame);
HAL_StatusTypeDef ETH_ReadData(ETH_Handle_t *handle, void **buffer, uint32_t *length);
```

#### Configuration and Status
```c
HAL_StatusTypeDef ETH_SetMACAddress(ETH_Handle_t *handle, uint8_t *macAddr);
void ETH_GetMACAddress(ETH_Handle_t *handle, uint8_t *macAddr);
bool ETH_IsLinkUp(ETH_Handle_t *handle);
uint32_t ETH_GetLinkSpeed(ETH_Handle_t *handle);
uint32_t ETH_GetLinkDuplex(ETH_Handle_t *handle);
```

#### Interrupt Management
```c
HAL_StatusTypeDef ETH_EnableInterrupts(ETH_Handle_t *handle);
HAL_StatusTypeDef ETH_DisableInterrupts(ETH_Handle_t *handle);
void ETH_IRQHandler(ETH_Handle_t *handle);
```

#### PHY Management
```c
HAL_StatusTypeDef ETH_ReadPHYRegister(uint16_t phyAddr, uint16_t regAddr, uint32_t *regValue);
HAL_StatusTypeDef ETH_WritePHYRegister(uint16_t phyAddr, uint16_t regAddr, uint32_t regValue);
HAL_StatusTypeDef ETH_InitPHY(ETH_Handle_t *handle);
HAL_StatusTypeDef ETH_AutoNegotiate(ETH_Handle_t *handle);
```

#### Callback Functions
```c
void ETH_TxCpltCallback(ETH_HandleTypeDef *heth);     // Transmission complete
void ETH_RxCpltCallback(ETH_HandleTypeDef *heth);     // Reception complete
void ETH_ErrorCallback(ETH_HandleTypeDef *heth);      // Error occurred
void ETH_WakeUpCallback(ETH_HandleTypeDef *heth);     // Wake-up event
```

### Constants and Macros

```c
// Ethernet frame constants
#define ETH_ADDR_LEN             6U
#define ETH_HEADER_LEN           14U
#define ETH_MIN_FRAME_LEN        60U
#define ETH_MAX_FRAME_LEN        1518U

// Ethernet protocol types
#define ETH_TYPE_ARP             0x0806U
#define ETH_TYPE_IPV4            0x0800U
#define ETH_TYPE_IPV6            0x86DDU
#define ETH_TYPE_VLAN            0x8100U

// Speed and duplex modes
#define ETH_SPEED_10M            0x00000000U
#define ETH_SPEED_100M           0x00004000U
#define ETH_MODE_HALFDUPLEX      0x00000000U
#define ETH_MODE_FULLDUPLEX      0x00000100U

// Media interfaces
#define ETH_MEDIA_INTERFACE_MII  0x00000000U
#define ETH_MEDIA_INTERFACE_RMII 0x00000001U

// Checksum modes
#define ETH_CHECKSUM_BY_HARDWARE 0x00000000U
#define ETH_CHECKSUM_BY_SOFTWARE 0x00000001U

// Special MAC addresses
#define ETH_MAC_ADDR_BROADCAST   {0xFF, 0xFF, 0xFF, 0xFF, 0xFF, 0xFF}
#define ETH_MAC_ADDR_MULTICAST   {0x01, 0x00, 0x5E, 0x00, 0x00, 0x00}
```

### Error Codes

| Error Code | Description |
|------------|-------------|
| HAL_OK | Operation successful |
| HAL_ERROR | General error |
| HAL_BUSY | Peripheral busy |
| HAL_TIMEOUT | Operation timeout |
| HAL_INVALID_PARAM | Invalid parameter |
| HAL_ETH_ERROR_PARAM | Parameter error |
| HAL_ETH_ERROR_BUSY | Ethernet busy |
| HAL_ETH_ERROR_TIMEOUT | Ethernet timeout |

## Basic Usage Examples

### Example 1: Basic Ethernet Initialization

```c
#include "Peripherals/ETH/eth.h"

int main(void) {
    // Initialize system
    HAL_Init();
    
    // Configure Ethernet
    ETH_Config_t ethConfig = {
        .macAddr = {0x00, 0x11, 0x22, 0x33, 0x44, 0x55},  // MAC address
        .speed = ETH_SPEED_100M,                           // 100 Mbps
        .duplexMode = ETH_MODE_FULLDUPLEX,                 // Full duplex
        .checksumMode = ETH_CHECKSUM_BY_HARDWARE,          // Hardware checksum
        .mediaInterface = ETH_MEDIA_INTERFACE_RMII,        // RMII interface
        .vlanTagIdentifier = 0,                            // No VLAN
        .vlanTagProtocol = 0
    };
    
    // Create Ethernet handle
    ETH_Handle_t ethHandle;
    
    // Allocate buffers
    uint8_t txBuffer[ETH_MAX_FRAME_LEN];
    uint8_t rxBuffer[ETH_MAX_FRAME_LEN];
    
    ethHandle.rxBuffer = rxBuffer;
    ethHandle.rxBufferSize = sizeof(rxBuffer);
    ethHandle.txBuffer = txBuffer;
    ethHandle.txBufferSize = sizeof(txBuffer);
    
    // Initialize Ethernet
    if (ETH_Init(&ethHandle, &ethConfig) != HAL_OK) {
        // Handle initialization error
        while(1);
    }
    
    // Start Ethernet communication
    if (ETH_Start(&ethHandle) != HAL_OK) {
        // Handle start error
        while(1);
    }
    
    // Enable interrupts
    ETH_EnableInterrupts(&ethHandle);
    
    // Ethernet is now ready for communication
    while(1) {
        // Main application loop
    }
}
```

### Example 2: Sending an Ethernet Frame

```c
// Send a simple Ethernet frame
void send_test_frame(ETH_Handle_t *ethHandle) {
    ETH_Frame_t frame;
    
    // Set destination MAC (broadcast)
    memset(frame.destination, 0xFF, 6);
    
    // Set source MAC
    ETH_GetMACAddress(ethHandle, frame.source);
    
    // Set Ethernet type (example protocol)
    frame.type = 0x8888;  // Example protocol
    
    // Prepare payload
    uint8_t payload[] = "Hello Ethernet!";
    frame.payload = payload;
    frame.payloadLength = strlen((char*)payload);
    
    // Send frame
    if (ETH_TransmitFrame(ethHandle, &frame) == HAL_OK) {
        printf("Frame sent successfully\n");
    } else {
        printf("Failed to send frame\n");
    }
}
```

### Example 3: Receiving Ethernet Frames

```c
// Receive and process Ethernet frames
void process_received_frames(ETH_Handle_t *ethHandle) {
    ETH_Frame_t receivedFrame;
    
    // Check if frame is available
    if (ETH_ReceiveFrame(ethHandle, &receivedFrame) == HAL_OK) {
        // Process frame based on type
        switch (receivedFrame.type) {
            case ETH_TYPE_ARP:
                printf("Received ARP frame\n");
                // Handle ARP packet
                break;
                
            case ETH_TYPE_IPV4:
                printf("Received IPv4 frame\n");
                // Handle IPv4 packet
                break;
                
            default:
                printf("Received unknown frame type: 0x%04X\n", receivedFrame.type);
                break;
        }
        
        // Print frame information
        printf("Source MAC: %02X:%02X:%02X:%02X:%02X:%02X\n",
               receivedFrame.source[0], receivedFrame.source[1],
               receivedFrame.source[2], receivedFrame.source[3],
               receivedFrame.source[4], receivedFrame.source[5]);
        
        printf("Payload length: %lu bytes\n", receivedFrame.payloadLength);
    }
}
```

### Example 4: ARP Request Implementation

```c
// Send ARP request to resolve IP address
void send_arp_request(ETH_Handle_t *ethHandle, uint8_t *targetIP) {
    ETH_Frame_t arpFrame;
    
    // Broadcast destination
    memset(arpFrame.destination, 0xFF, 6);
    
    // Source MAC
    ETH_GetMACAddress(ethHandle, arpFrame.source);
    
    // ARP protocol
    arpFrame.type = ETH_TYPE_ARP;
    
    // ARP payload (simplified)
    uint8_t arpPayload[28] = {
        0x00, 0x01, // Hardware type: Ethernet
        0x08, 0x00, // Protocol type: IPv4
        0x06,       // Hardware address length
        0x04,       // Protocol address length
        0x00, 0x01, // Operation: ARP request
        // Sender MAC
        arpFrame.source[0], arpFrame.source[1], arpFrame.source[2],
        arpFrame.source[3], arpFrame.source[4], arpFrame.source[5],
        // Sender IP (example: 192.168.1.100)
        192, 168, 1, 100,
        // Target MAC (unknown)
        0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
        // Target IP
        targetIP[0], targetIP[1], targetIP[2], targetIP[3]
    };
    
    arpFrame.payload = arpPayload;
    arpFrame.payloadLength = sizeof(arpPayload);
    
    // Send ARP request
    ETH_TransmitFrame(ethHandle, &arpFrame);
}
```

### Example 5: Interrupt-Driven Communication

```c
// Global variables for interrupt handling
volatile bool ethTxComplete = false;
volatile bool ethRxComplete = false;
volatile bool ethError = false;

// Ethernet interrupt handler
void ETH_IRQHandler(void) {
    ETH_IRQHandler(&ethHandle);
}

// Transmission complete callback
void ETH_TxCpltCallback(ETH_HandleTypeDef *heth) {
    ethTxComplete = true;
    // Signal application
    osSignalSet(ethTaskHandle, ETH_TX_COMPLETE_SIGNAL);
}

// Reception complete callback
void ETH_RxCpltCallback(ETH_HandleTypeDef *heth) {
    ethRxComplete = true;
    // Signal application
    osSignalSet(ethTaskHandle, ETH_RX_COMPLETE_SIGNAL);
}

// Error callback
void ETH_ErrorCallback(ETH_HandleTypeDef *heth) {
    ethError = true;
    uint32_t error = HAL_ETH_GetError(heth);
    printf("Ethernet error: 0x%08lX\n", error);
}

// Interrupt-driven frame transmission
void send_frame_interrupt(ETH_Handle_t *ethHandle, ETH_Frame_t *frame) {
    ethTxComplete = false;
    
    if (ETH_TransmitFrame_IT(ethHandle, frame) == HAL_OK) {
        // Wait for completion or handle in RTOS
        while (!ethTxComplete) {
            // Could yield to RTOS here
        }
    }
}
```

## Advanced Features

### PHY Management and Auto-negotiation

```c
// PHY register definitions (example for LAN8720)
#define PHY_BCR                0x00    // Basic Control Register
#define PHY_BSR                0x01    // Basic Status Register
#define PHY_ID1                0x02    // PHY Identifier 1
#define PHY_ID2                0x03    // PHY Identifier 2
#define PHY_ANAR               0x04    // Auto-Negotiation Advertisement
#define PHY_ANLPAR             0x05    // Auto-Negotiation Link Partner Ability
#define PHY_ANER               0x06    // Auto-Negotiation Expansion

// PHY control bits
#define PHY_RESET              (1 << 15)
#define PHY_LOOPBACK           (1 << 14)
#define PHY_SPEED_100          (1 << 13)
#define PHY_AUTONEG_ENABLE     (1 << 12)
#define PHY_POWER_DOWN         (1 << 11)
#define PHY_RESTART_AUTONEG    (1 << 9)

// Initialize and configure PHY
HAL_StatusTypeDef init_phy(ETH_Handle_t *ethHandle) {
    uint32_t regValue;
    uint16_t phyAddr = 0;  // PHY address (usually 0 or 1)
    
    // Read PHY identifier
    ETH_ReadPHYRegister(phyAddr, PHY_ID1, &regValue);
    printf("PHY ID1: 0x%04X\n", regValue);
    
    ETH_ReadPHYRegister(phyAddr, PHY_ID2, &regValue);
    printf("PHY ID2: 0x%04X\n", regValue);
    
    // Reset PHY
    ETH_WritePHYRegister(phyAddr, PHY_BCR, PHY_RESET);
    HAL_Delay(100);  // Wait for reset to complete
    
    // Enable auto-negotiation
    ETH_WritePHYRegister(phyAddr, PHY_BCR, PHY_AUTONEG_ENABLE);
    
    // Restart auto-negotiation
    ETH_WritePHYRegister(phyAddr, PHY_BCR, PHY_AUTONEG_ENABLE | PHY_RESTART_AUTONEG);
    
    // Wait for auto-negotiation to complete
    uint32_t timeout = 5000;  // 5 seconds timeout
    do {
        ETH_ReadPHYRegister(phyAddr, PHY_BSR, &regValue);
        HAL_Delay(1);
        timeout--;
    } while (!(regValue & (1 << 5)) && timeout > 0);  // Check auto-neg complete bit
    
    if (timeout == 0) {
        return HAL_TIMEOUT;
    }
    
    // Read link status
    ETH_ReadPHYRegister(phyAddr, PHY_BSR, &regValue);
    if (regValue & (1 << 2)) {  // Link up
        printf("Link is up\n");
        
        // Read link partner ability to determine speed/duplex
        ETH_ReadPHYRegister(phyAddr, PHY_ANLPAR, &regValue);
        
        if (regValue & (1 << 8)) {
            printf("100 Mbps Full Duplex\n");
        } else if (regValue & (1 << 7)) {
            printf("100 Mbps Half Duplex\n");
        } else if (regValue & (1 << 6)) {
            printf("10 Mbps Full Duplex\n");
        } else {
            printf("10 Mbps Half Duplex\n");
        }
    } else {
        printf("Link is down\n");
    }
    
    return HAL_OK;
}
```

### VLAN Support

```c
// Configure VLAN tagging
HAL_StatusTypeDef configure_vlan(ETH_Handle_t *ethHandle, uint16_t vlanId, uint16_t vlanProtocol) {
    // Enable VLAN tag insertion
    ETH->MACVLANTR = vlanId | (1 << 17);  // Enable VLAN tag comparison
    
    // Configure VLAN protocol identifier
    ethHandle->config.vlanTagIdentifier = vlanId;
    ethHandle->config.vlanTagProtocol = vlanProtocol;
    
    return HAL_OK;
}

// Send VLAN-tagged frame
void send_vlan_frame(ETH_Handle_t *ethHandle, uint16_t vlanId) {
    ETH_Frame_t frame;
    
    // Set destination MAC
    memset(frame.destination, 0xFF, 6);
    ETH_GetMACAddress(ethHandle, frame.source);
    
    // VLAN-tagged frame (802.1Q)
    frame.type = 0x8100;  // VLAN tag protocol identifier
    
    // VLAN payload: [TPID(2)][TCI(2)][Original Type(2)][Data...]
    uint8_t vlanPayload[1500];
    
    // TPID (Tag Protocol Identifier) - already set in type
    // TCI (Tag Control Information): PCP(3) | DEI(1) | VID(12)
    uint16_t tci = vlanId & 0x0FFF;  // VLAN ID (12 bits)
    vlanPayload[0] = (tci >> 8) & 0xFF;
    vlanPayload[1] = tci & 0xFF;
    
    // Original Ethernet type (e.g., IPv4)
    vlanPayload[2] = 0x08;
    vlanPayload[3] = 0x00;
    
    // Copy actual payload
    uint8_t data[] = "VLAN Tagged Data";
    memcpy(&vlanPayload[4], data, sizeof(data));
    
    frame.payload = vlanPayload;
    frame.payloadLength = 4 + sizeof(data);  // VLAN header + data
    
    ETH_TransmitFrame(ethHandle, &frame);
}
```

### Multicast and Broadcast Filtering

```c
// Configure multicast filtering
HAL_StatusTypeDef configure_multicast_filter(ETH_Handle_t *ethHandle, uint8_t *multicastAddr) {
    // Calculate hash value for multicast address
    uint32_t hashValue = calculate_crc32(multicastAddr, 6);
    uint32_t hashBit = (hashValue >> 26) & 0x3F;  // Use upper 6 bits
    
    if (hashBit < 32) {
        // Set bit in hash table low register
        ETH->MACHTLR |= (1 << hashBit);
    } else {
        // Set bit in hash table high register
        ETH->MACHTHR |= (1 << (hashBit - 32));
    }
    
    // Enable multicast hash filtering
    ETH->MACFFR |= ETH_MACFFR_HM;
    
    return HAL_OK;
}

// Enable broadcast reception
void enable_broadcast_reception(ETH_Handle_t *ethHandle) {
    // Clear broadcast frames disable bit
    ETH->MACFFR &= ~ETH_MACFFR_BFD;
}

// Enable promiscuous mode
void enable_promiscuous_mode(ETH_Handle_t *ethHandle) {
    ETH->MACFFR |= ETH_MACFFR_PM;
}
```

### Flow Control (Pause Frames)

```c
// Configure flow control
HAL_StatusTypeDef configure_flow_control(ETH_Handle_t *ethHandle, bool txFlowControl, bool rxFlowControl) {
    uint32_t flowControl = 0;
    
    if (txFlowControl) {
        flowControl |= ETH_MACFCR_TFE;  // Transmit flow control enable
    }
    
    if (rxFlowControl) {
        flowControl |= ETH_MACFCR_RFE;  // Receive flow control enable
    }
    
    // Set pause time (number of slot times)
    flowControl |= (0x100 << 16);  // 256 slot times pause
    
    ETH->MACFCR = flowControl;
    
    return HAL_OK;
}

// Send pause frame manually
void send_pause_frame(ETH_Handle_t *ethHandle, uint16_t pauseTime) {
    // Configure pause time
    ETH->MACFCR &= ~(0xFFFF << 16);
    ETH->MACFCR |= (pauseTime << 16);
    
    // Trigger pause frame transmission
    ETH->MACFCR |= ETH_MACFCR_FCB_BPA;  // Back pressure activate
    
    // Wait for transmission
    HAL_Delay(1);
    
    // Clear back pressure
    ETH->MACFCR &= ~ETH_MACFCR_FCB_BPA;
}
```

### Jumbo Frame Support

```c
// Configure jumbo frame support
HAL_StatusTypeDef configure_jumbo_frames(ETH_Handle_t *ethHandle, uint16_t maxFrameSize) {
    if (maxFrameSize > 9000) {
        return HAL_INVALID_PARAM;  // Too large
    }
    
    // Disable jabber timer (allows longer frames)
    ETH->MACCR |= ETH_MACCR_JD;
    
    // Configure DMA to accept larger frames
    // Note: This requires custom DMA descriptor configuration
    
    // Update buffer sizes
    ethHandle->rxBufferSize = maxFrameSize;
    ethHandle->txBufferSize = maxFrameSize;
    
    return HAL_OK;
}
```

### Power Management

```c
// Enter power-down mode
HAL_StatusTypeDef enter_power_down(ETH_Handle_t *ethHandle) {
    // Stop Ethernet
    ETH_Stop(ethHandle);
    
    // Enable power down mode
    ETH->MACPMTCSR |= ETH_MACPMTCSR_PD;
    
    return HAL_OK;
}

// Configure wake-on-LAN
HAL_StatusTypeDef configure_wake_on_lan(ETH_Handle_t *ethHandle, uint8_t *wolPattern) {
    // Enable magic packet wake-up
    ETH->MACPMTCSR |= ETH_MACPMTCSR_MPE;
    
    // Configure wake-up frame filter (simplified)
    // This would require setting up the wake-up frame filter registers
    
    return HAL_OK;
}
```

## Network Protocols

### ARP (Address Resolution Protocol)

```c
// ARP packet structure
typedef struct {
    uint16_t hardwareType;      // Hardware type (1 = Ethernet)
    uint16_t protocolType;      // Protocol type (0x0800 = IPv4)
    uint8_t hardwareLen;        // Hardware address length (6)
    uint8_t protocolLen;        // Protocol address length (4)
    uint16_t operation;         // Operation (1=request, 2=reply)
    uint8_t senderMac[6];       // Sender MAC address
    uint8_t senderIP[4];        // Sender IP address
    uint8_t targetMac[6];       // Target MAC address
    uint8_t targetIP[4];        // Target IP address
} arp_packet_t;

// Send ARP request
void arp_request(ETH_Handle_t *ethHandle, uint8_t *targetIP) {
    ETH_Frame_t frame;
    arp_packet_t arp;
    
    // Broadcast destination
    memset(frame.destination, 0xFF, 6);
    ETH_GetMACAddress(ethHandle, frame.source);
    frame.type = ETH_TYPE_ARP;
    
    // Fill ARP packet
    arp.hardwareType = htons(1);      // Ethernet
    arp.protocolType = htons(0x0800); // IPv4
    arp.hardwareLen = 6;
    arp.protocolLen = 4;
    arp.operation = htons(1);         // Request
    
    // Sender information
    memcpy(arp.senderMac, frame.source, 6);
    uint8_t myIP[] = {192, 168, 1, 100};  // Example IP
    memcpy(arp.senderIP, myIP, 4);
    
    // Target information
    memset(arp.targetMac, 0, 6);      // Unknown
    memcpy(arp.targetIP, targetIP, 4);
    
    frame.payload = (uint8_t*)&arp;
    frame.payloadLength = sizeof(arp_packet_t);
    
    ETH_TransmitFrame(ethHandle, &frame);
}

// Process ARP reply
void process_arp_reply(uint8_t *payload, uint32_t length) {
    if (length < sizeof(arp_packet_t)) return;
    
    arp_packet_t *arp = (arp_packet_t*)payload;
    
    if (ntohs(arp->operation) == 2) {  // Reply
        printf("ARP Reply: IP %d.%d.%d.%d -> MAC %02X:%02X:%02X:%02X:%02X:%02X\n",
               arp->senderIP[0], arp->senderIP[1], arp->senderIP[2], arp->senderIP[3],
               arp->senderMac[0], arp->senderMac[1], arp->senderMac[2],
               arp->senderMac[3], arp->senderMac[4], arp->senderMac[5]);
        
        // Update ARP cache
        // arp_cache_add(arp->senderIP, arp->senderMac);
    }
}
```

### IPv4 Packet Handling

```c
// IPv4 header structure
typedef struct {
    uint8_t versionIHL;         // Version (4) and IHL (header length)
    uint8_t tos;                // Type of service
    uint16_t totalLength;       // Total length
    uint16_t identification;    // Identification
    uint16_t flagsFragment;     // Flags and fragment offset
    uint8_t ttl;                // Time to live
    uint8_t protocol;           // Protocol (TCP=6, UDP=17, ICMP=1)
    uint16_t headerChecksum;    // Header checksum
    uint8_t sourceIP[4];        // Source IP address
    uint8_t destIP[4];          // Destination IP address
} ipv4_header_t;

// Calculate IP header checksum
uint16_t calculate_ip_checksum(uint8_t *data, uint16_t length) {
    uint32_t sum = 0;
    
    for (uint16_t i = 0; i < length; i += 2) {
        sum += (data[i] << 8) | data[i + 1];
    }
    
    sum = (sum >> 16) + (sum & 0xFFFF);
    sum += (sum >> 16);
    
    return ~sum;
}

// Process IPv4 packet
void process_ipv4_packet(uint8_t *payload, uint32_t length) {
    if (length < sizeof(ipv4_header_t)) return;
    
    ipv4_header_t *ip = (ipv4_header_t*)payload;
    
    // Convert from network byte order
    uint16_t totalLen = ntohs(ip->totalLength);
    uint8_t headerLen = (ip->versionIHL & 0x0F) * 4;
    
    printf("IPv4 Packet: %d.%d.%d.%d -> %d.%d.%d.%d, Protocol: %d, Length: %d\n",
           ip->sourceIP[0], ip->sourceIP[1], ip->sourceIP[2], ip->sourceIP[3],
           ip->destIP[0], ip->destIP[1], ip->destIP[2], ip->destIP[3],
           ip->protocol, totalLen);
    
    // Process payload based on protocol
    uint8_t *ipPayload = payload + headerLen;
    uint16_t payloadLen = totalLen - headerLen;
    
    switch (ip->protocol) {
        case 1:  // ICMP
            process_icmp_packet(ipPayload, payloadLen);
            break;
        case 6:  // TCP
            process_tcp_packet(ipPayload, payloadLen);
            break;
        case 17: // UDP
            process_udp_packet(ipPayload, payloadLen);
            break;
    }
}
```

### ICMP (Ping) Implementation

```c
// ICMP header structure
typedef struct {
    uint8_t type;               // Type (8=echo request, 0=echo reply)
    uint8_t code;               // Code (0 for echo)
    uint16_t checksum;          // Checksum
    uint16_t identifier;        // Identifier
    uint16_t sequence;          // Sequence number
    uint8_t data[];             // Data payload
} icmp_header_t;

// Calculate ICMP checksum
uint16_t calculate_icmp_checksum(uint8_t *data, uint16_t length) {
    return calculate_ip_checksum(data, length);
}

// Send ping (ICMP echo request)
void send_ping(ETH_Handle_t *ethHandle, uint8_t *destIP, uint8_t *destMac) {
    ETH_Frame_t frame;
    
    // Ethernet header
    memcpy(frame.destination, destMac, 6);
    ETH_GetMACAddress(ethHandle, frame.source);
    frame.type = ETH_TYPE_IPV4;
    
    // Create IPv4 + ICMP packet
    uint8_t packet[100];
    ipv4_header_t *ip = (ipv4_header_t*)packet;
    icmp_header_t *icmp = (icmp_header_t*)(packet + 20);
    
    // IPv4 header
    ip->versionIHL = 0x45;      // Version 4, header length 5
    ip->tos = 0;
    ip->totalLength = htons(20 + 8 + 32);  // IP + ICMP + data
    ip->identification = 0;
    ip->flagsFragment = 0;
    ip->ttl = 64;
    ip->protocol = 1;           // ICMP
    ip->headerChecksum = 0;
    uint8_t myIP[] = {192, 168, 1, 100};
    memcpy(ip->sourceIP, myIP, 4);
    memcpy(ip->destIP, destIP, 4);
    ip->headerChecksum = calculate_ip_checksum((uint8_t*)ip, 20);
    
    // ICMP header
    icmp->type = 8;             // Echo request
    icmp->code = 0;
    icmp->checksum = 0;
    icmp->identifier = htons(1);
    icmp->sequence = htons(1);
    
    // ICMP data
    uint8_t *icmpData = icmp->data;
    memcpy(icmpData, "Hello from STM32!", 18);
    
    // Calculate ICMP checksum
    icmp->checksum = calculate_icmp_checksum((uint8_t*)icmp, 8 + 18);
    
    frame.payload = packet;
    frame.payloadLength = 20 + 8 + 18;  // IP + ICMP + data
    
    ETH_TransmitFrame(ethHandle, &frame);
}

// Process ICMP packet
void process_icmp_packet(uint8_t *payload, uint32_t length) {
    if (length < sizeof(icmp_header_t)) return;
    
    icmp_header_t *icmp = (icmp_header_t*)payload;
    
    if (icmp->type == 8) {  // Echo request
        printf("Received ping request\n");
        // Send echo reply
    } else if (icmp->type == 0) {  // Echo reply
        printf("Received ping reply\n");
    }
}
```

### Simple TCP Server

```c
// Simplified TCP header
typedef struct {
    uint16_t sourcePort;        // Source port
    uint16_t destPort;          // Destination port
    uint32_t sequenceNum;       // Sequence number
    uint32_t ackNum;            // Acknowledgment number
    uint8_t dataOffset;         // Data offset and flags
    uint8_t flags;              // TCP flags
    uint16_t window;            // Window size
    uint16_t checksum;          // Checksum
    uint16_t urgentPtr;         // Urgent pointer
} tcp_header_t;

// TCP connection state
typedef enum {
    TCP_CLOSED,
    TCP_LISTEN,
    TCP_SYN_RECEIVED,
    TCP_ESTABLISHED,
    TCP_CLOSE_WAIT,
    TCP_LAST_ACK,
    TCP_FIN_WAIT_1,
    TCP_FIN_WAIT_2,
    TCP_TIME_WAIT
} tcp_state_t;

// Simple TCP server for port 80 (HTTP)
void tcp_server_task(ETH_Handle_t *ethHandle) {
    tcp_state_t tcpState = TCP_LISTEN;
    uint32_t seqNum = 0x12345678;
    uint32_t ackNum = 0;
    
    while (1) {
        ETH_Frame_t frame;
        
        if (ETH_ReceiveFrame(ethHandle, &frame) == HAL_OK) {
            if (frame.type == ETH_TYPE_IPV4) {
                ipv4_header_t *ip = (ipv4_header_t*)frame.payload;
                if (ip->protocol == 6) {  // TCP
                    tcp_header_t *tcp = (tcp_header_t*)(frame.payload + 20);
                    uint16_t destPort = ntohs(tcp->destPort);
                    
                    if (destPort == 80) {  // HTTP port
                        switch (tcpState) {
                            case TCP_LISTEN:
                                if (tcp->flags & 0x02) {  // SYN flag
                                    // Send SYN-ACK
                                    send_tcp_syn_ack(ethHandle, frame.source, ip->sourceIP, tcp);
                                    tcpState = TCP_SYN_RECEIVED;
                                }
                                break;
                                
                            case TCP_ESTABLISHED:
                                if (tcp->flags & 0x10) {  // ACK flag
                                    // Process HTTP request
                                    process_http_request(frame.payload + 20 + 20, frame.payloadLength - 40);
                                }
                                break;
                        }
                    }
                }
            }
        }
        
        HAL_Delay(10);
    }
}
```

## Performance Considerations

### Ethernet Performance Characteristics

| Configuration | Throughput | CPU Usage | Latency |
|---------------|------------|-----------|---------|
| 100Mbps Full Duplex | 100 Mbps | 5-15% | <50μs |
| 10Mbps Full Duplex | 10 Mbps | 2-8% | <100μs |
| Hardware Checksum | +20% throughput | -5% CPU | Same |
| Interrupt Mode | Same | <2% | <10μs |
| Polling Mode | Same | 10-20% | <5μs |

### Optimization Techniques

1. **DMA Buffer Configuration**
   ```c
   // Optimize buffer sizes for your application
   #define ETH_RX_BUFFER_SIZE  1536  // Jumbo frame support
   #define ETH_TX_BUFFER_SIZE  1536
   #define ETH_RX_DESC_COUNT   4     // Reduce for low latency
   #define ETH_TX_DESC_COUNT   4
   
   // Use aligned buffers for better performance
   uint8_t rxBuffers[ETH_RX_DESC_COUNT][ETH_RX_BUFFER_SIZE] __attribute__((aligned(4)));
   uint8_t txBuffers[ETH_TX_DESC_COUNT][ETH_TX_BUFFER_SIZE] __attribute__((aligned(4)));
   ```

2. **Interrupt Optimization**
   ```c
   // Configure interrupt priorities
   NVIC_SetPriority(ETH_IRQn, NVIC_EncodePriority(NVIC_GetPriorityGrouping(), 1, 0));
   
   // Use interrupt preemption for critical packets
   // Enable only necessary interrupts
   ETH->DMAIER = ETH_DMAIER_NISE | ETH_DMAIER_RIE | ETH_DMAIER_TIE;
   ```

3. **PHY Optimization**
   ```c
   // Force speed/duplex for better performance
   ETH_WritePHYRegister(phyAddr, PHY_BCR, PHY_SPEED_100 | PHY_FULLDUPLEX);
   
   // Disable auto-negotiation if speed is known
   // This reduces link establishment time
   ```

4. **Memory Optimization**
   ```c
   // Place Ethernet buffers in fast memory
   uint8_t ethBuffers[ETH_BUFFER_SIZE] __attribute__((section(".fast_ram")));
   
   // Use DMA-capable memory regions
   // Avoid cacheable regions that require frequent invalidation
   ```

5. **Protocol Optimization**
   ```c
   // Enable hardware checksum offload
   ETH->MACCR |= ETH_MACCR_IPCO;
   
   // Use appropriate frame sizes
   // Larger frames reduce overhead but increase latency
   ```

### Memory Usage
- **ETH_Handle_t**: ~200 bytes
- **HAL Handle**: ~150 bytes
- **RX Descriptors**: 16 bytes × descriptor count
- **TX Descriptors**: 16 bytes × descriptor count
- **Buffers**: Configurable (typically 1-4KB each)
- **Total**: 2-8KB depending on configuration

### CPU Utilization
- **Interrupt Processing**: <5% for normal traffic
- **Polling Mode**: 10-20% for high traffic
- **PHY Management**: <1% (periodic)
- **Protocol Processing**: Depends on stack complexity
- **DMA Overhead**: Minimal (hardware accelerated)

### Bandwidth Considerations
- **Ethernet MAC**: 100 Mbps maximum
- **AHB Bus**: 168 MHz (672 MB/s theoretical)
- **Memory Bandwidth**: Depends on SDRAM/SRAM speed
- **PHY Interface**: RMII (9 signals) or MII (18 signals)
- **Buffer Copying**: Avoid when possible (use zero-copy)

### Latency Optimization
- **Interrupt Priority**: High priority for time-critical packets
- **Buffer Pre-allocation**: Avoid dynamic allocation in ISR
- **Descriptor Count**: Fewer descriptors reduce latency
- **Cache Management**: Proper cache handling for DMA buffers
- **PHY Configuration**: Optimize link parameters

## Troubleshooting

### Common Ethernet Issues

#### 1. No Link Detected

**Problem**: ETH_IsLinkUp() returns false

**Solutions**:
- Check PHY connections (TX/RX pairs, clock)
- Verify PHY address (0 or 1 typically)
- Check PHY power supply (3.3V)
- Verify RMII/MII mode configuration
- Check crystal oscillator (25MHz for RMII)
- Test with different cable/partner

**Debug Code**:
```c
void debug_link_status(ETH_Handle_t *ethHandle) {
    uint32_t regValue;
    uint16_t phyAddr = 0;
    
    // Read PHY status register
    ETH_ReadPHYRegister(phyAddr, PHY_BSR, &regValue);
    printf("PHY BSR: 0x%04X\n", regValue);
    
    if (regValue & (1 << 2)) {
        printf("PHY Link Up\n");
    } else {
        printf("PHY Link Down\n");
    }
    
    // Check auto-negotiation status
    if (regValue & (1 << 5)) {
        printf("Auto-negotiation Complete\n");
    } else {
        printf("Auto-negotiation In Progress\n");
    }
    
    // Read link partner ability
    ETH_ReadPHYRegister(phyAddr, PHY_ANLPAR, &regValue);
    printf("Link Partner: 0x%04X\n", regValue);
}
```

#### 2. Cannot Transmit Frames

**Problem**: ETH_TransmitFrame() returns error or timeout

**Solutions**:
- Verify Ethernet initialization completed successfully
- Check TX descriptor availability
- Verify buffer alignment (32-bit)
- Check frame size (60-1518 bytes)
- Ensure proper Ethernet header format
- Verify DMA is enabled and configured
- Check for TX buffer overflow

**TX Debug**:
```c
void debug_transmission(ETH_Handle_t *ethHandle) {
    // Check DMA status
    printf("DMA SR: 0x%08X\n", ETH->DMASR);
    
    // Check MAC configuration
    printf("MAC CR: 0x%08X\n", ETH->MACCR);
    
    // Check if transmitter is enabled
    if (ETH->MACCR & ETH_MACCR_TE) {
        printf("Transmitter enabled\n");
    } else {
        printf("Transmitter disabled\n");
    }
    
    // Check DMA transmit status
    if (ETH->DMASR & ETH_DMASR_TS) {
        printf("Transmit suspended\n");
        ETH->DMASR = ETH_DMASR_TS;  // Clear flag
    }
    
    // Check descriptor status
    ETH_DMADescTypeDef *txDesc = (ETH_DMADescTypeDef*)ETH->DMATDLAR;
    printf("TX Desc Status: 0x%08X\n", txDesc->Status);
}
```

#### 3. Cannot Receive Frames

**Problem**: ETH_ReceiveFrame() never receives data

**Solutions**:
- Verify receiver is enabled (ETH_MACCR_RE)
- Check RX descriptor configuration
- Verify buffer size and alignment
- Check MAC address filtering
- Ensure proper interrupt configuration
- Verify DMA RX is enabled
- Check for RX buffer overflow

**RX Debug**:
```c
void debug_reception(ETH_Handle_t *ethHandle) {
    // Check DMA status
    printf("DMA SR: 0x%08X\n", ETH->DMASR);
    
    // Check if receiver is enabled
    if (ETH->MACCR & ETH_MACCR_RE) {
        printf("Receiver enabled\n");
    } else {
        printf("Receiver disabled\n");
    }
    
    // Check MAC frame filter
    printf("MAC FFR: 0x%08X\n", ETH->MACFFR);
    
    // Check RX descriptor status
    ETH_DMADescTypeDef *rxDesc = (ETH_DMADescTypeDef*)ETH->DMARDLAR;
    printf("RX Desc Status: 0x%08X\n", rxDesc->Status);
    
    // Check if frame is available
    if (rxDesc->Status & ETH_DMARXDESC_OWN) {
        printf("RX descriptor owned by DMA\n");
    } else {
        printf("Frame available in RX descriptor\n");
        uint32_t frameLength = rxDesc->Status >> 16;
        printf("Frame length: %d bytes\n", frameLength);
    }
}
```

#### 4. Data Corruption

**Problem**: Received data is corrupted

**Solutions**:
- Check buffer alignment (32-bit aligned)
- Verify cache management (invalidate after DMA)
- Check for memory corruption by other peripherals
- Verify DMA burst size configuration
- Check for bus contention
- Verify buffer size matches descriptor configuration

**Memory Debug**:
```c
void debug_memory_corruption(uint8_t *buffer, uint32_t size) {
    // Check buffer alignment
    uintptr_t addr = (uintptr_t)buffer;
    if (addr & 0x03) {
        printf("Buffer not 32-bit aligned: 0x%08X\n", addr);
    }
    
    // Fill buffer with known pattern
    memset(buffer, 0xAA, size);
    
    // Check if pattern is preserved
    for (uint32_t i = 0; i < size; i++) {
        if (buffer[i] != 0xAA) {
            printf("Memory corruption at offset %d\n", i);
            break;
        }
    }
    
    // Test cache operations
    SCB_CleanDCache_by_Addr((uint32_t*)buffer, size);
    SCB_InvalidateDCache_by_Addr((uint32_t*)buffer, size);
}
```

#### 5. Auto-negotiation Issues

**Problem**: Link fails to establish or wrong speed/duplex

**Solutions**:
- Force specific speed/duplex for testing
- Check cable quality and length
- Verify PHY supports auto-negotiation
- Check link partner capabilities
- Reset PHY and retry negotiation
- Use external PHY configuration tools

**PHY Debug**:
```c
void debug_phy_status(uint16_t phyAddr) {
    uint32_t regValue;
    
    // Read all important PHY registers
    const char *regNames[] = {"BCR", "BSR", "PHYID1", "PHYID2", "ANAR", "ANLPAR"};
    uint8_t regAddrs[] = {0x00, 0x01, 0x02, 0x03, 0x04, 0x05};
    
    for (int i = 0; i < 6; i++) {
        ETH_ReadPHYRegister(phyAddr, regAddrs[i], &regValue);
        printf("PHY %s (0x%02X): 0x%04X\n", regNames[i], regAddrs[i], regValue);
    }
    
    // Interpret status
    ETH_ReadPHYRegister(phyAddr, PHY_BSR, &regValue);
    if (regValue & (1 << 2)) {
        printf("Link: UP\n");
        
        // Determine speed/duplex from ANLPAR
        ETH_ReadPHYRegister(phyAddr, PHY_ANLPAR, &regValue);
        if (regValue & (1 << 8)) printf("100Mbps Full Duplex\n");
        else if (regValue & (1 << 7)) printf("100Mbps Half Duplex\n");
        else if (regValue & (1 << 6)) printf("10Mbps Full Duplex\n");
        else printf("10Mbps Half Duplex\n");
    } else {
        printf("Link: DOWN\n");
    }
}
```

#### 6. Interrupt Issues

**Problem**: Interrupts not triggering or wrong callbacks called

**Solutions**:
- Verify NVIC configuration and priority
- Check interrupt enable bits in DMAIER/MACIMR
- Verify ISR is installed correctly
- Check for interrupt sharing conflicts
- Verify callback functions are implemented
- Check interrupt flags in status registers

**Interrupt Debug**:
```c
void debug_interrupts(void) {
    // Check NVIC settings
    printf("ETH_IRQn enabled: %s\n", NVIC_GetEnableIRQ(ETH_IRQn) ? "Yes" : "No");
    uint32_t priority = NVIC_GetPriority(ETH_IRQn);
    printf("ETH_IRQn priority: %d\n", priority);
    
    // Check DMA interrupt enables
    printf("DMA IER: 0x%08X\n", ETH->DMAIER);
    printf("MAC IMR: 0x%08X\n", ETH->MACIMR);
    
    // Check interrupt status
    printf("DMA SR: 0x%08X\n", ETH->DMASR);
    printf("MAC SR: 0x%08X\n", ETH->MACISR);
    
    // Clear any pending interrupts
    ETH->DMASR = 0xFFFFFFFF;  // Clear all DMA interrupts
    ETH->MACISR = 0xFFFFFFFF; // Clear all MAC interrupts
}
```

### Hardware Verification

```c
// Comprehensive Ethernet hardware test
typedef struct {
    const char *name;
    HAL_StatusTypeDef (*test_func)(void);
} eth_test_t;

HAL_StatusTypeDef test_ethernet_initialization(void) {
    ETH_Config_t config = {
        .macAddr = {0x00, 0x11, 0x22, 0x33, 0x44, 0x55},
        .speed = ETH_SPEED_100M,
        .duplexMode = ETH_MODE_FULLDUPLEX,
        .mediaInterface = ETH_MEDIA_INTERFACE_RMII
    };
    
    ETH_Handle_t handle;
    uint8_t rxBuf[1520], txBuf[1520];
    handle.rxBuffer = rxBuf;
    handle.txBuffer = txBuf;
    handle.rxBufferSize = sizeof(rxBuf);
    handle.txBufferSize = sizeof(txBuf);
    
    HAL_StatusTypeDef status = ETH_Init(&handle, &config);
    if (status != HAL_OK) return status;
    
    status = ETH_Start(&handle);
    if (status != HAL_OK) {
        ETH_DeInit(&handle);
        return status;
    }
    
    // Basic register check
    if (!(ETH->MACCR & ETH_MACCR_TE) || !(ETH->MACCR & ETH_MACCR_RE)) {
        ETH_Stop(&handle);
        ETH_DeInit(&handle);
        return HAL_ERROR;
    }
    
    ETH_Stop(&handle);
    ETH_DeInit(&handle);
    return HAL_OK;
}

HAL_StatusTypeDef test_loopback_transmission(void) {
    // Configure MAC for internal loopback
    ETH->MACCR |= ETH_MACCR_LM;
    
    ETH_Frame_t txFrame, rxFrame;
    uint8_t testData[] = "Loopback Test";
    
    // Prepare frame
    memset(txFrame.destination, 0xFF, 6);
    ETH_GetMACAddress(NULL, txFrame.source);
    txFrame.type = 0x8888;
    txFrame.payload = testData;
    txFrame.payloadLength = strlen((char*)testData);
    
    // Send frame
    HAL_StatusTypeDef status = ETH_TransmitFrame(NULL, &txFrame);
    if (status != HAL_OK) return status;
    
    HAL_Delay(10);  // Allow time for loopback
    
    // Try to receive
    status = ETH_ReceiveFrame(NULL, &rxFrame);
    if (status != HAL_OK) return status;
    
    // Verify data
    if (rxFrame.payloadLength != txFrame.payloadLength) {
        return HAL_ERROR;
    }
    
    if (memcmp(rxFrame.payload, txFrame.payload, rxFrame.payloadLength) != 0) {
        return HAL_ERROR;
    }
    
    // Disable loopback
    ETH->MACCR &= ~ETH_MACCR_LM;
    return HAL_OK;
}

void run_ethernet_tests(void) {
    eth_test_t tests[] = {
        {"Ethernet Initialization", test_ethernet_initialization},
        {"Loopback Transmission", test_loopback_transmission},
        // Add more tests...
    };
    
    printf("=== Ethernet Hardware Test Suite ===\n");
    
    for (int i = 0; i < sizeof(tests)/sizeof(tests[0]); i++) {
        printf("Running %s... ", tests[i].name);
        HAL_StatusTypeDef result = tests[i].test_func();
        printf("%s\n", result == HAL_OK ? "PASS" : "FAIL");
    }
    
    printf("=== Tests Complete ===\n");
}
```

### Network Analysis Tools

```c
// Ethernet traffic analyzer
typedef struct {
    uint32_t framesReceived;
    uint32_t framesTransmitted;
    uint32_t bytesReceived;
    uint32_t bytesTransmitted;
    uint32_t crcErrors;
    uint32_t alignmentErrors;
    uint32_t overruns;
    uint32_t collisions;
} eth_statistics_t;

void collect_ethernet_stats(eth_statistics_t *stats) {
    // Read MMC counters (if enabled)
    stats->framesReceived = ETH->MMCRGUFCR;    // Good unicast frames
    stats->crcErrors = ETH->MMCRFCECR;         // CRC errors
    stats->alignmentErrors = ETH->MMCRFAECR;   // Alignment errors
    
    // Read other counters as needed
    // Note: MMC must be enabled in MACCR
}

void print_network_stats(eth_statistics_t *stats) {
    printf("=== Ethernet Statistics ===\n");
    printf("Frames RX: %lu\n", stats->framesReceived);
    printf("Frames TX: %lu\n", stats->framesTransmitted);
    printf("Bytes RX: %lu\n", stats->bytesReceived);
    printf("Bytes TX: %lu\n", stats->bytesTransmitted);
    printf("CRC Errors: %lu\n", stats->crcErrors);
    printf("Alignment Errors: %lu\n", stats->alignmentErrors);
    printf("Overruns: %lu\n", stats->overruns);
    printf("Collisions: %lu\n", stats->collisions);
    
    // Calculate error rates
    if (stats->framesReceived > 0) {
        float errorRate = (float)(stats->crcErrors + stats->alignmentErrors) / stats->framesReceived * 100.0f;
        printf("Error Rate: %.2f%%\n", errorRate);
    }
}
```

## Summary

The STM32F429 Ethernet peripheral provides comprehensive networking capabilities with hardware acceleration for high-performance embedded applications. Key takeaways:

### Hardware Features
- IEEE 802.3 compliant 10/100 Mbps MAC with full-duplex support
- Hardware checksum offload for TCP/UDP/IP protocols
- Flexible RMII/MII PHY interface with auto-negotiation
- DMA-based data transfer with configurable buffer descriptors
- Advanced filtering (unicast, multicast, broadcast, VLAN)
- Flow control with pause frame generation
- Power management with Wake-on-LAN support
- Jumbo frame support (up to 16KB)

### Software Features
- HAL-based API with comprehensive error handling
- Polling and interrupt-driven operation modes
- Flexible frame buffer management
- Protocol stack support (ARP, IPv4, ICMP, TCP, UDP)
- Network statistics and monitoring
- Multicast and broadcast filtering
- VLAN tagging and priority handling
- Link status monitoring and auto-negotiation

### Performance Characteristics
- 100 Mbps full-duplex Ethernet throughput
- Hardware-accelerated protocol processing
- Low CPU overhead (5-15% for typical applications)
- Sub-50μs interrupt latency
- Efficient memory usage (2-8KB for buffers)
- DMA-based zero-copy operations

### Best Practices
1. Use hardware checksum offload to reduce CPU load
2. Configure appropriate buffer sizes for your application
3. Implement proper interrupt handling for real-time response
4. Use auto-negotiation for automatic link configuration
5. Enable flow control for reliable high-speed communication
6. Implement proper cache management for DMA buffers
7. Use VLAN tagging for network segmentation
8. Monitor link status and handle disconnections gracefully
9. Implement network statistics for debugging and monitoring
10. Use appropriate filtering to reduce unnecessary interrupts

### Common Applications
- Embedded web servers and HTTP applications
- IoT devices with network connectivity
- Industrial Ethernet and automation systems
- Network monitoring and diagnostic tools
- Firmware update over network (TFTP)
- Remote monitoring and control systems
- Real-time data streaming applications
- Network protocol analyzers and testers
- VPN and secure communication devices
- Network-attached storage (NAS) systems

This tutorial provides comprehensive coverage of Ethernet usage on STM32F429, from basic frame transmission to advanced protocol implementation, with detailed troubleshooting and performance optimization techniques for robust network applications.